<a href="https://colab.research.google.com/github/inhajourney/AIFFEL_quest_rs/blob/main/kor_eng_seq2seq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 한국어 → 영어 번역기: Attention 기반 Seq2Seq 모델

## 프로젝트 개요
- **소스 언어**: 한국어 (KoNLPy Mecab 토크나이저)
- **타겟 언어**: 영어 (공백 기반 + special token)
- **모델**: Bahdanau Attention + GRU 기반 Seq2Seq
- **데이터**: [jungyeul/korean-parallel-corpora](https://github.com/jungyeul/korean-parallel-corpora)

| 스텝 | 내용 |
|------|------|
| Step 0 | 라이브러리 import 및 환경 설정 |
| Step 1 | 데이터 다운로드 |
| Step 2 | 데이터 정제 (중복 제거 + 전처리 + 길이 필터링) |
| Step 3 | 데이터 토큰화 (Mecab / 공백 split) |
| Step 4 | 모델 설계 (Encoder / Attention / Decoder / Seq2Seq) |
| Step 5 | 훈련 + 예문 번역 출력 + Attention Map 시각화 |


## Step 0. 라이브러리 import 및 환경 설정

**하는 일**
- 필요한 모든 라이브러리를 불러옵니다.
- **matplotlib 한국어 폰트 설정**: NanumBarunGothic → NanumGothic → AppleGothic → 맑은고딕 순으로 자동 탐색
- GPU가 있으면 CUDA, 없으면 CPU를 자동 선택합니다.
- 재현성을 위해 랜덤 시드를 고정합니다.

**결과 출력**: PyTorch 버전, 사용 디바이스, 폰트 설정 결과


In [1]:
import os
import re
import random
import urllib.request
import tarfile
import logging

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.ticker as ticker

# ── 한국어 폰트 설정 (깨짐 방지) ──
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

_FONT_CANDIDATES = [
    "/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf",  # 리눅스 나눔폰트
    "/usr/share/fonts/truetype/nanum/NanumGothic.ttf",
    "/System/Library/Fonts/AppleSDGothicNeo.ttc",             # macOS
    "C:/Windows/Fonts/malgun.ttf",                            # Windows 맑은고딕
]

_font_set = False
for _fp in _FONT_CANDIDATES:
    if os.path.exists(_fp):
        _fontprop = fm.FontProperties(fname=_fp, size=12)
        plt.rcParams["font.family"] = _fontprop.get_name()
        plt.rcParams["axes.unicode_minus"] = False  # 마이너스 기호 깨짐 방지
        print(f"  ✅ 한국어 폰트 설정 완료: {_fontprop.get_name()}  ({_fp})")
        _font_set = True
        break

if not _font_set:
    print("  ⚠️  폰트를 찾지 못했습니다. sudo apt-get install -y fonts-nanum 후 재실행")

# ── 시드 고정 ──
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# ── 디바이스 설정 ──
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"  PyTorch 버전  : {torch.__version__}")
print(f"  사용 디바이스 : {device}")
if torch.cuda.is_available():
    print(f"  GPU 이름      : {torch.cuda.get_device_name(0)}")


  ⚠️  폰트를 찾지 못했습니다. sudo apt-get install -y fonts-nanum 후 재실행
  PyTorch 버전  : 2.10.0+cpu
  사용 디바이스 : cpu


## Step 1. 데이터 다운로드

**하는 일**
- [korean-parallel-corpora](https://github.com/jungyeul/korean-parallel-corpora) 레포에서
  `korean-english-park.train.tar.gz` 파일을 다운로드합니다.
- 압축을 풀어 한국어(`.ko`)와 영어(`.en`) 파일을 얻습니다.
- 각 줄이 하나의 문장 → **같은 줄 번호가 병렬 쌍**

**결과 출력**: 한국어/영어 문장 수, 샘플 쌍


In [6]:
DATA_DIR = "./kor_eng_data"
os.makedirs(DATA_DIR, exist_ok=True)

KO_PATH = os.path.join(DATA_DIR, "korean-english-park.train.ko")
EN_PATH = os.path.join(DATA_DIR, "korean-english-park.train.en")

BASE = "https://raw.githubusercontent.com/haven-jeon/ko_en_neural_machine_translation/master/korean_parallel_corpora/korean-english-v1"
KO_URL = f"{BASE}/korean-english-park.train.ko"
EN_URL = f"{BASE}/korean-english-park.train.en"

if not os.path.exists(KO_PATH):
    print("  한국어 데이터 다운로드 중...")
    urllib.request.urlretrieve(KO_URL, KO_PATH)
    print("  완료!")

if not os.path.exists(EN_PATH):
    print("  영어 데이터 다운로드 중...")
    urllib.request.urlretrieve(EN_URL, EN_PATH)
    print("  완료!")

with open(KO_PATH, encoding="utf-8") as f:
    ko_lines = f.readlines()
with open(EN_PATH, encoding="utf-8") as f:
    en_lines = f.readlines()

print(f"  한국어 문장 수: {len(ko_lines):,}")
print(f"  영어  문장 수 : {len(en_lines):,}")
print(f"  샘플 (한): {ko_lines[0].strip()}")
print(f"  샘플 (영): {en_lines[0].strip()}")

  한국어 데이터 다운로드 중...
  완료!
  영어 데이터 다운로드 중...
  완료!
  한국어 문장 수: 96,215
  영어  문장 수 : 96,215
  샘플 (한): 개인용 컴퓨터 사용의 상당 부분은 "이것보다 뛰어날 수 있느냐?"
  샘플 (영): Much of personal computing is about "can you top this?"


## Step 2. 데이터 정제

**하는 일**

① **전처리 함수 정의**
- 영문: 소문자화 + 구두점 분리 + 영문/구두점 외 제거
- 한글: `[^가-힣0-9\s]` 정규식으로 한글/숫자/공백만 유지

② **중복 제거** (`set` 활용)
- `(한글, 영어)` 튜플 단위로 set에 넣어 중복 제거
- 병렬 관계가 흐트러지지 않도록 **쌍(pair) 단위** 처리

③ **토큰 길이 필터링** (≤ 40)
- 너무 긴 문장은 학습에 방해 → 양쪽 모두 40 이하인 쌍만 선별

**결과**: `cleaned_corpus`, `kor_corpus`, `eng_corpus`


In [7]:
# ① 전처리 함수 정의
def preprocess_english(sentence: str) -> str:
    """영어 문장 전처리: 소문자화 + 특수문자 정리"""
    sentence = sentence.lower().strip()
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)
    sentence = re.sub(r'[" "]+', " ", sentence)
    sentence = re.sub(r"[^a-zA-Z?.!,]+", " ", sentence)
    return sentence.strip()

def preprocess_korean(sentence: str) -> str:
    """한국어 문장 전처리: 한글/숫자/공백만 유지"""
    sentence = sentence.strip()
    sentence = re.sub(r"[^가-힣0-9\s]", " ", sentence)
    sentence = re.sub(r"\s+", " ", sentence)
    return sentence.strip()

# ② 중복 제거 (set 활용 — 병렬 쌍 단위로 처리)
raw_pairs = list(zip(
    [ko.strip() for ko in ko_lines],
    [en.strip() for en in en_lines]
))
print(f"  원본 데이터 쌍 수  : {len(raw_pairs):,}")

cleaned_corpus = list({
    (preprocess_korean(ko), preprocess_english(en))
    for ko, en in raw_pairs
    if ko.strip() and en.strip()
})
print(f"  중복 제거 후 쌍 수 : {len(cleaned_corpus):,}")

# ③ 토큰 길이 필터링 (≤ 40)
filtered = [
    (ko, en) for ko, en in cleaned_corpus
    if len(ko.split()) <= 40 and len(en.split()) <= 40
]
print(f"  길이 40 이하 필터링 후: {len(filtered):,}")

kor_corpus = [pair[0] for pair in filtered]
eng_corpus = [pair[1] for pair in filtered]

print(f"\n  샘플 (정제된 한): {kor_corpus[0]}")
print(f"  샘플 (정제된 영): {eng_corpus[0]}")


  원본 데이터 쌍 수  : 96,215
  중복 제거 후 쌍 수 : 80,911
  길이 40 이하 필터링 후: 74,127

  샘플 (정제된 한): 박신원
  샘플 (정제된 영): the judge rejected the argument , noting


## Step 3. 데이터 토큰화

**하는 일**
- **한국어**: KoNLPy Mecab으로 형태소 분석 (없으면 공백 split으로 자동 대체)
- **영어**: `<start>` + 공백 split + `<end>` 추가
- **Vocabulary 구축**: 단어 → 인덱스 딕셔너리 (vocab 크기 최소 10,000 권장)
  - 특수 토큰: `<pad>=0`, `<unk>=1`, `<start>=2`, `<end>=3`
- **`tokenize()`**: 문장 리스트 → 패딩된 LongTensor 변환

**결과 출력**: vocab 크기, 샘플 토큰, 텐서 shape


In [8]:
# KoNLPy Mecab 로드 (없으면 공백 split fallback)
try:
    from konlpy.tag import Mecab
    mecab = Mecab()
    def tokenize_korean(sentence: str):
        return mecab.morphs(sentence)
    print("  ✅ KoNLPy Mecab 사용")
except Exception:
    def tokenize_korean(sentence: str):
        return sentence.split()
    print("  ⚠️  Mecab 없음 → 공백 split으로 대체")

def tokenize_english(sentence: str):
    """영어 토큰화: <start> + split + <end>"""
    return ["<start>"] + sentence.split() + ["<end>"]

print("  한국어 토크나이징 중...")
kor_tokenized = [tokenize_korean(s) for s in kor_corpus]
print("  영어 토크나이징 중...")
eng_tokenized = [tokenize_english(s) for s in eng_corpus]

print(f"  샘플 한국어 토큰: {kor_tokenized[0]}")
print(f"  샘플 영어  토큰: {eng_tokenized[0]}")


  ⚠️  Mecab 없음 → 공백 split으로 대체
  한국어 토크나이징 중...
  영어 토크나이징 중...
  샘플 한국어 토큰: ['박신원']
  샘플 영어  토큰: ['<start>', 'the', 'judge', 'rejected', 'the', 'argument', ',', 'noting', '<end>']


In [9]:
from collections import Counter

def build_vocab(tokenized_corpus, min_freq=1):
    """
    단어 빈도를 세어 vocab 딕셔너리를 구축합니다.
    특수 토큰: <pad>=0, <unk>=1, <start>=2, <end>=3
    """
    counter = Counter(token for sent in tokenized_corpus for token in sent)
    vocab = {"<pad>": 0, "<unk>": 1, "<start>": 2, "<end>": 3}
    for word, freq in counter.most_common():
        if freq >= min_freq and word not in vocab:
            vocab[word] = len(vocab)
    return vocab

kor_vocab = build_vocab(kor_tokenized, min_freq=1)
eng_vocab = build_vocab(eng_tokenized, min_freq=1)

print(f"  한국어 vocab 크기: {len(kor_vocab):,}")
print(f"  영어   vocab 크기: {len(eng_vocab):,}")
if len(kor_vocab) < 10000:
    print("  ⚠️  vocab이 10,000 미만입니다. 데이터를 늘리거나 min_freq를 낮추세요.")

kor_idx2word = {v: k for k, v in kor_vocab.items()}
eng_idx2word = {v: k for k, v in eng_vocab.items()}


  한국어 vocab 크기: 183,958
  영어   vocab 크기: 41,896


In [10]:
def tokenize(tokenized_sentences, vocab, max_len=50):
    """
    토큰화된 문장 리스트를 LongTensor로 변환합니다.
    max_len 이상은 잘라내고, 부족하면 <pad>로 채웁니다.
    """
    PAD_ID = vocab["<pad>"]
    UNK_ID = vocab["<unk>"]
    result = []
    for tokens in tokenized_sentences:
        ids = [vocab.get(t, UNK_ID) for t in tokens[:max_len]]
        ids += [PAD_ID] * (max_len - len(ids))
        result.append(ids)
    return torch.LongTensor(result)

MAX_LEN = 50
kor_tensor = tokenize(kor_tokenized, kor_vocab, MAX_LEN)
eng_tensor = tokenize(eng_tokenized, eng_vocab, MAX_LEN)

print(f"  한국어 텐서 shape: {kor_tensor.shape}  → (문장 수, max_len)")
print(f"  영어   텐서 shape: {eng_tensor.shape}")


  한국어 텐서 shape: torch.Size([74127, 50])  → (문장 수, max_len)
  영어   텐서 shape: torch.Size([74127, 50])


## Step 4. 모델 설계

**구성 요소**

| 클래스 | 역할 |
|--------|------|
| `TranslationDataset` | enc_input / dec_input / dec_label 분리 |
| `BahdanauAttention` | score = v·tanh(W1·h + W2·s) |
| `Encoder` | GRU로 한국어 → hidden state 시퀀스 |
| `Decoder` | Attention + GRU → 매 스텝 영어 단어 예측 |
| `Seq2SeqAttention` | 학습: Teacher Forcing / 추론: Greedy Decoding |

**하이퍼파라미터**
- Embedding Size: 256
- Hidden Size: 512
- Dropout: 0.3

**결과 출력**: 모델 구조 및 학습 가능 파라미터 수


In [11]:
class TranslationDataset(Dataset):
    """
    한→영 병렬 코퍼스 Dataset.
    enc_input : 한국어 인덱스 텐서 (max_len,)
    dec_input : <start> ~ 마지막 단어  (teacher forcing 입력)
    dec_label : 첫 단어 ~ <end>        (예측 대상)
    """
    def __init__(self, kor_tensor, eng_tensor):
        self.kor = kor_tensor
        self.eng = eng_tensor

    def __len__(self):
        return len(self.kor)

    def __getitem__(self, idx):
        enc_input = self.kor[idx]
        dec_input = self.eng[idx][:-1]  # <start> ~ 마지막
        dec_label = self.eng[idx][1:]   # 첫 단어 ~ <end>
        return enc_input, dec_input, dec_label


In [12]:
class BahdanauAttention(nn.Module):
    """
    Bahdanau (Additive) Attention.
    score(s, h) = v^T · tanh(W1·h + W2·s)

    입력:
      hidden          : (batch, hid_dim)
      encoder_outputs : (src_len, batch, hid_dim)
    출력:
      attention weight: (batch, src_len)  — softmax 적용됨
    """
    def __init__(self, hid_dim):
        super().__init__()
        self.W1 = nn.Linear(hid_dim, hid_dim)
        self.W2 = nn.Linear(hid_dim, hid_dim)
        self.v  = nn.Linear(hid_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs):
        src_len = encoder_outputs.shape[0]
        hidden = hidden.unsqueeze(1).repeat(1, src_len, 1)
        encoder_outputs = encoder_outputs.permute(1, 0, 2)
        energy = torch.tanh(self.W1(encoder_outputs) + self.W2(hidden))
        attention = self.v(energy).squeeze(2)
        return torch.softmax(attention, dim=1)


In [13]:
class Encoder(nn.Module):
    """
    GRU 기반 인코더.
    입력  src     : (src_len, batch)
    출력  outputs : (src_len, batch, hid_dim)
          hidden  : (1, batch, hid_dim)
    """
    def __init__(self, input_dim, emb_dim, hid_dim, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim, padding_idx=0)
        self.rnn       = nn.GRU(emb_dim, hid_dim, batch_first=False)
        self.dropout   = nn.Dropout(dropout)

    def forward(self, src):
        embedded = self.dropout(self.embedding(src))
        outputs, hidden = self.rnn(embedded)
        return outputs, hidden


class Decoder(nn.Module):
    """
    Attention + GRU 기반 디코더.
    RNN 입력: embedding + context vector 결합
    출력층 : hidden state + context vector 결합 → 다음 토큰 예측
    """
    def __init__(self, output_dim, emb_dim, hid_dim, attention, dropout=0.3):
        super().__init__()
        self.attention = attention
        self.embedding = nn.Embedding(output_dim, emb_dim, padding_idx=0)
        self.rnn     = nn.GRU(emb_dim + hid_dim, hid_dim, batch_first=False)
        self.fc_out  = nn.Linear(hid_dim * 2, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, encoder_outputs):
        input    = input.unsqueeze(0)
        embedded = self.dropout(self.embedding(input))
        a = self.attention(hidden[-1], encoder_outputs)     # (batch, src_len)
        a = a.unsqueeze(1)                                  # (batch, 1, src_len)
        enc = encoder_outputs.permute(1, 0, 2)              # (batch, src_len, hid)
        context = torch.bmm(a, enc).permute(1, 0, 2)        # (1, batch, hid)
        rnn_input = torch.cat((embedded, context), dim=2)
        output, hidden = self.rnn(rnn_input, hidden)
        output  = output.squeeze(0)
        context = context.squeeze(0)
        prediction = self.fc_out(torch.cat((output, context), dim=1))
        return prediction, hidden, a.squeeze(1)


In [14]:
class Seq2SeqAttention(nn.Module):
    """
    학습 모드 (trg 제공) : Teacher Forcing
    추론 모드 (trg=None) : Greedy Decoding (<start> 토큰부터 시작)
    """
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device  = device

    def forward(self, src, trg=None, max_len=50, teacher_forcing_ratio=0.5):
        batch_size = src.shape[1]
        BOS_ID, EOS_ID = 2, 3
        encoder_outputs, hidden = self.encoder(src)
        outputs, attentions = [], []

        if trg is not None:
            # 학습 모드: Teacher Forcing
            input = trg[0]
            for t in range(1, trg.shape[0]):
                output, hidden, attn = self.decoder(input, hidden, encoder_outputs)
                outputs.append(output.unsqueeze(0))
                attentions.append(attn.unsqueeze(0))
                use_teacher = random.random() < teacher_forcing_ratio
                input = trg[t] if use_teacher else output.argmax(1)
        else:
            # 추론 모드: Greedy Decoding
            input    = torch.full((batch_size,), BOS_ID, dtype=torch.long, device=self.device)
            finished = torch.zeros(batch_size, dtype=torch.bool, device=self.device)
            for _ in range(max_len):
                output, hidden, attn = self.decoder(input, hidden, encoder_outputs)
                outputs.append(output.unsqueeze(0))
                attentions.append(attn.unsqueeze(0))
                top1 = output.argmax(1)
                input = top1
                finished |= (top1 == EOS_ID)
                if finished.all():
                    break

        outputs    = torch.cat(outputs, dim=0)
        attentions = torch.cat(attentions, dim=0)
        return outputs, attentions


# 하이퍼파라미터 & 모델 초기화
INPUT_DIM  = len(kor_vocab)
OUTPUT_DIM = len(eng_vocab)
EMB_DIM    = 256
HID_DIM    = 512
DROPOUT    = 0.3

attention = BahdanauAttention(HID_DIM).to(device)
encoder   = Encoder(INPUT_DIM, EMB_DIM, HID_DIM, DROPOUT).to(device)
decoder   = Decoder(OUTPUT_DIM, EMB_DIM, HID_DIM, attention, DROPOUT).to(device)
model     = Seq2SeqAttention(encoder, decoder, device).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Encoder  입력 vocab : {INPUT_DIM:,}")
print(f"  Decoder  출력 vocab : {OUTPUT_DIM:,}")
print(f"  Embedding 크기      : {EMB_DIM}")
print(f"  Hidden 크기         : {HID_DIM}")
print(f"  학습 가능 파라미터  : {total_params:,}")
print(model)


  Encoder  입력 vocab : 183,958
  Decoder  출력 vocab : 41,896
  Embedding 크기      : 256
  Hidden 크기         : 512
  학습 가능 파라미터  : 104,439,720
Seq2SeqAttention(
  (encoder): Encoder(
    (embedding): Embedding(183958, 256, padding_idx=0)
    (rnn): GRU(256, 512)
    (dropout): Dropout(p=0.3, inplace=False)
  )
  (decoder): Decoder(
    (attention): BahdanauAttention(
      (W1): Linear(in_features=512, out_features=512, bias=True)
      (W2): Linear(in_features=512, out_features=512, bias=True)
      (v): Linear(in_features=512, out_features=1, bias=False)
    )
    (embedding): Embedding(41896, 256, padding_idx=0)
    (rnn): GRU(768, 512)
    (fc_out): Linear(in_features=1024, out_features=41896, bias=True)
    (dropout): Dropout(p=0.3, inplace=False)
  )
)


## Step 5. 훈련하기

**하는 일**
- DataLoader 구성 (batch_size=64)
- Adam optimizer + CrossEntropyLoss (`<pad>` 인덱스 무시)
- 그래디언트 클리핑 (max_norm=1) — exploding gradient 방지
- **매 에폭마다**: Loss 출력 + 예문 4개 번역 결과 출력
- **훈련 완료 후**: Attention Heat Map 시각화

**예문**
```
K1) 오바마는 대통령이다.
K2) 시민들은 도시 속에 산다.
K3) 커피는 필요 없다.
K4) 일곱 명의 사망자가 발생했다.
```


In [15]:
dataset    = TranslationDataset(kor_tensor, eng_tensor)
BATCH_SIZE = 64
loader     = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

PAD_ID    = eng_vocab["<pad>"]
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)

print(f"  전체 샘플 수 : {len(dataset):,}")
print(f"  배치 수      : {len(loader):,}  (batch_size={BATCH_SIZE})")
print("  슝~")


  전체 샘플 수 : 74,127
  배치 수      : 1,159  (batch_size=64)
  슝~


In [16]:
def translate(sentence_ko: str, max_len: int = 50) -> str:
    """한국어 문장 → 영어 번역 문자열 반환"""
    model.eval()
    with torch.no_grad():
        tokens = tokenize_korean(preprocess_korean(sentence_ko))
        ids    = [kor_vocab.get(t, kor_vocab["<unk>"]) for t in tokens[:max_len]]
        ids   += [kor_vocab["<pad>"]] * (max_len - len(ids))
        src    = torch.LongTensor(ids).unsqueeze(1).to(device)

        outputs, _ = model(src, trg=None, max_len=max_len)
        pred_ids   = outputs.squeeze(1).argmax(1).tolist()

        result = []
        for idx in pred_ids:
            word = eng_idx2word.get(idx, "<unk>")
            result.append(word)
            if word == "<end>":
                break
    return " ".join(result)

# 예문 정의
test_sentences = [
    "오바마는 대통령이다.",
    "시민들은 도시 속에 산다.",
    "커피는 필요 없다.",
    "일곱 명의 사망자가 발생했다.",
]
print("번역 함수 준비 완료! 슝~")


번역 함수 준비 완료! 슝~


In [ ]:
NUM_EPOCHS = 20
print(f"에폭 수: {NUM_EPOCHS}")
print("-" * 60)

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    epoch_loss = 0

    for enc_input, dec_input, dec_label in loader:
        enc_input = enc_input.permute(1, 0).to(device)
        dec_input = dec_input.permute(1, 0).to(device)
        dec_label = dec_label.permute(1, 0).to(device)

        optimizer.zero_grad()
        outputs, _ = model(enc_input, trg=dec_input)

        trg_len  = outputs.shape[0]                    # ← 핵심
        out_flat = outputs.reshape(-1, OUTPUT_DIM)
        lbl_flat = dec_label[:trg_len].reshape(-1)     # ← 핵심

        loss = criterion(out_flat, lbl_flat)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1)
        optimizer.step()
        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(loader)
    print(f"[Epoch {epoch:02d}/{NUM_EPOCHS}]  Loss: {avg_loss:.4f}")

    for i, sent in enumerate(test_sentences, 1):
        print(f"  K{i}) {sent}  →  {translate(sent)}")
    print()

에폭 수: 20
------------------------------------------------------------


## Attention Map 시각화

어텐션 맵을 보면 **디코더가 각 영어 단어를 생성할 때 어느 한국어 토큰에 집중했는지** 시각적으로 확인할 수 있습니다.

- **x축**: 한국어 입력 토큰
- **y축**: 영어 출력 토큰
- **색이 진할수록** 해당 한국어 토큰에 많이 집중


In [ ]:
def plot_attention(sentence_ko: str, max_len: int = 50):
    """번역 + Attention Heat Map 시각화"""
    model.eval()
    with torch.no_grad():
        src_tokens = tokenize_korean(preprocess_korean(sentence_ko))
        ids        = [kor_vocab.get(t, kor_vocab["<unk>"]) for t in src_tokens[:max_len]]
        ids       += [kor_vocab["<pad>"]] * (max_len - len(ids))
        src        = torch.LongTensor(ids).unsqueeze(1).to(device)
        outputs, attentions = model(src, trg=None, max_len=max_len)
        pred_ids   = outputs.squeeze(1).argmax(1).tolist()

    trg_tokens = []
    for idx in pred_ids:
        word = eng_idx2word.get(idx, "<unk>")
        trg_tokens.append(word)
        if word == "<end>":
            break

    attn = attentions[:len(trg_tokens), 0, :len(src_tokens)].cpu().numpy()

    fig, ax = plt.subplots(figsize=(10, 6))
    cax = ax.matshow(attn, cmap="YlOrRd")
    fig.colorbar(cax)
    ax.set_xticklabels([""] + src_tokens, rotation=45, ha="left", fontsize=10)
    ax.set_yticklabels([""] + trg_tokens, fontsize=10)
    ax.xaxis.set_major_locator(ticker.MultipleLocator(1))
    ax.yaxis.set_major_locator(ticker.MultipleLocator(1))
    ax.set_title(f"Attention Map\n입력: {sentence_ko}", fontsize=12, pad=20)
    ax.set_xlabel("한국어 입력 토큰")
    ax.set_ylabel("영어 출력 토큰")
    plt.tight_layout()
    plt.show()

# 예문 전체 Attention Map 출력
for sent in test_sentences:
    plot_attention(sent)


## 최종 번역 결과

학습이 끝난 모델로 예문 4개를 번역합니다.


In [1]:
print("=" * 60)
print("최종 번역 결과")
print("=" * 60)
for i, sent in enumerate(test_sentences, 1):
    print(f"K{i}) {sent}")
    print(f"E{i}) {translate(sent)}")
    print()
print("모든 스텝 완료! 🎉")


최종 번역 결과


NameError: name 'test_sentences' is not defined